# Steel Plant Dispatch — Exploratory Data Analysis
This notebook explores an anonymised dispatch register. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / 'data' / 'dispatch_data.csv')
df['Production Date'] = pd.to_datetime(df['Production Date'], errors='coerce')
df.head()

## 1. Structure and data quality

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all').T)
quality = pd.DataFrame({'missing': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(2)})
display(quality.sort_values('missing', ascending=False))

## 2. Headline KPIs
Dispatch value uses the row-level dispatch-value field. Do not sum the source Total NSR field.

In [ ]:
kpis = {
    'Total dispatch (MT)': df['Qty/Weight (t)'].sum(),
    'Dispatch value (Rs)': df['Dispatch Value (Rs)'].sum(),
    'Dispatch lines': len(df),
    'Unique customers': df['Customer Name'].nunique(),
    'Unique sales orders': df['Sales Order'].nunique()
}
pd.Series(kpis)

## 3. Monthly dispatch trend

In [ ]:
df['Year-Month'] = df['Production Date'].dt.to_period('M').astype(str)
monthly = df.groupby('Year-Month')['Qty/Weight (t)'].sum().sort_index()
ax = monthly.plot(kind='line', marker='o', figsize=(11,5), title='Monthly Dispatch (MT)')
ax.set_xlabel('Month'); ax.set_ylabel('Dispatch (MT)'); ax.grid(axis='y', alpha=.25)
plt.xticks(rotation=35); plt.tight_layout(); plt.show()

## 4. Product and customer mix

In [ ]:
product = df.groupby('FG Product')['Qty/Weight (t)'].sum().sort_values()
product.plot(kind='barh', figsize=(9,5), title='Dispatch by Product (MT)')
plt.xlabel('Dispatch (MT)'); plt.tight_layout(); plt.show()

customer = df.groupby('Customer Name')['Qty/Weight (t)'].sum().nlargest(10).sort_values()
customer.plot(kind='barh', figsize=(9,5), title='Top 10 Customers by Dispatch (MT)')
plt.xlabel('Dispatch (MT)'); plt.tight_layout(); plt.show()

## 5. Shift, quality and destination

In [ ]:
for col in ['Shift', 'Quality', 'Destination']:
    display(df.groupby(col, dropna=False)['Qty/Weight (t)'].sum().sort_values(ascending=False).head(15))

## 6. Weighted commercial rates
A weighted average gives more influence to dispatch lines with more tonnes.

In [ ]:
def weighted_avg(data, rate_col):
    x = data[[rate_col, 'Qty/Weight (t)']].dropna()
    x = x[x['Qty/Weight (t)'] > 0]
    return (x[rate_col] * x['Qty/Weight (t)']).sum() / x['Qty/Weight (t)'].sum()

for col in ['NSR (Rs/t)', 'UPR (Rs/t)', 'Freight (Rs/t)', 'Discount (Rs/t)']:
    print(col, round(weighted_avg(df, col), 2))

## 7. Interpretation and limitations
Do not infer OTIF, on-time delivery, order fill rate or production-plan adherence from this register alone. Confirm the meaning of `Interval` with the data owner before treating it as a time measure. Shift-wise tonnage is dispatch recorded by shift, not labour productivity.